In [30]:
import pandas as pd

# clean data path.
clean_path = r"../datasets/clean_data/"

# modeled data path.
modeled_path = "../datasets/modeled_data/"

In [31]:
# Read the tables.
orders = pd.read_parquet(f"{clean_path}orders.parquet", engine="pyarrow")
order_items = pd.read_parquet(f"{clean_path}order_items.parquet", engine="pyarrow")
order_reviews = pd.read_parquet(f"{clean_path}order_reviews.parquet", engine="pyarrow")
customers = pd.read_parquet(f"{clean_path}customers.parquet", engine="pyarrow")
products = pd.read_parquet(f"{clean_path}products.parquet", engine="pyarrow")
sellers = pd.read_parquet(f"{clean_path}sellers.parquet", engine="pyarrow")

In [32]:
tables = [orders, order_items, order_reviews, customers, products, sellers]
total_memory_befor_nomalization = 0

for table in tables:
    total_memory_befor_nomalization += table.memory_usage(deep=True).sum()

print(f"{total_memory_befor_nomalization:,.0f} Bytese")

99,742,060 Bytese


In [33]:
# Create a surrogate keys for dimention tables.
orders["order_key"] = range(1, len(orders) + 1)
customers["customer_key"] = range(1, len(customers) + 1)
products["product_key"] = range(1, len(products) + 1)
sellers["seller_key"] = range(1, len(sellers) + 1)

In [34]:
assert (orders["customer_id"].isin(customers["customer_id"])).all()

In [35]:
# Normalize customers and sellers by deriving a table for locations.
customers_location = (customers[["customer_city", "customer_state"]].
    rename(columns={"customer_city": "city",
                    "customer_state": "state"}))

sellers_location = (sellers[["seller_city", "seller_state"]].
    rename(columns={"seller_city": "city",
           "seller_state": "state"}))

location = pd.concat([customers_location, sellers_location], axis=0)

# Drop duplicates.
location.drop_duplicates(inplace=True)

# Create a surrogate key.
location["location_key"] = range(1, len(location) + 1)

location = location[["location_key", "city", "state"]]

In [36]:
# Move location key column to customers and sellers table.
customers = customers.merge(
    right=location,
    left_on=["customer_city", "customer_state"],
    right_on=["city", "state"],
    how="left"
)

sellers = sellers.merge(
    right=location,
    left_on=["seller_city", "seller_state"],
    right_on=["city", "state"],
    how="left"
)

sellers = sellers[["seller_id", "seller_key", "location_key"]]

In [37]:
# Move customer_unique_id column to orders table and make 1:M relationship to reduce redundancy.
orders = orders.merge(
    right=customers[["customer_id", "customer_key"]],
    on="customer_id",
    how="left"
)

In [38]:
# Drop duplicates from customers table after merge.
customers.drop(columns="customer_id", inplace=True)

# Rename the column.
customers.rename(columns={"customer_unique_id": "customer_id"}, inplace=True)

customers = customers[["customer_id", "customer_key", "location_key"]]

In [39]:
orders = orders.merge(
    right=order_reviews,
    on="order_id",
    how="left"
)

orders = orders[["order_id", "order_key", "customer_key","review_score",
                 "order_purchase_timestamp", "order_approved_at",
                 "order_delivered_carrier_date", "order_delivered_customer_date",
                 "order_estimated_delivery_date", "review_creation_date",
                 "review_answer_timestamp", "order_status"]]

In [40]:
# Model the fact table.

# Move the order_key column to the order_items table.
order_items = order_items.merge(
    right=orders[["order_id", "order_key"]],
    on="order_id",
    how="left"
)

# Move the product_key column to the order_items table.
order_items = order_items.merge(
    right=products[["product_id", "product_key"]],
    on="product_id",
    how="left"
)

# Move the seller_key column to the order_items table.
order_items = order_items.merge(
    right=sellers[["seller_id", "seller_key"]],
    on="seller_id",
    how="left"
)

order_items = order_items[["order_key", "product_key",
                           "seller_key", "order_item_id",
                           "price", "freight_value",
                           "shipping_limit_date"]]

In [41]:
orders.to_parquet(f"{modeled_path}dim_orders.parquet", index=False, engine="pyarrow")

In [42]:
products = products[["product_id", "product_key",
                     "product_name_length", "product_description_length",
                     "product_photos_qty"]]

In [43]:
customers.to_parquet(f"{modeled_path}dim_customers.parquet", index=False, engine="pyarrow")

In [44]:
sellers.to_parquet(f"{modeled_path}dim_sellers.parquet", index=False, engine="pyarrow")

In [45]:
products.to_parquet(f"{modeled_path}dim_products.parquet", index=False, engine="pyarrow")

In [46]:
order_items.to_parquet(f"{modeled_path}fact_sales.parquet", index=False, engine="pyarrow")

In [47]:
location.to_parquet(f"{modeled_path}dim_location.parquet", index=False, engine="pyarrow")

In [48]:
tables = [orders, order_items, customers, products, sellers, location]
total_memory_after_nomalization = 0

for table in tables:
    total_memory_after_nomalization += table.memory_usage(deep=True).sum()

print(f"{total_memory_after_nomalization:,.0f} Bytese")

34,856,124 Bytese


In [49]:
print(f"Saved Memory: {total_memory_befor_nomalization - total_memory_after_nomalization:,.0f} Bytese")

Saved Memory: 64,885,936 Bytese
